# SkyAware Flight Delay Prediction — Notebook 8: CI/CD Pipeline

**AAI-540 MLOps | Group 4**

Defines a complete **SageMaker Pipeline** for automated model retraining and deployment.
The pipeline implements 7 checkpoints that gate model promotion from raw data to production:

```
Raw Data (S3)
     │
┌────▼──────────────────┐
│ Step 1: Preprocess    │  Feature engineering + temporal split
└────┬──────────────────┘
     │
┌────▼──────────────────┐
│ Step 2: Train         │  XGBoost SageMaker Training Job
└────┬──────────────────┘
     │
┌────▼──────────────────┐
│ Step 3: Evaluate      │  Batch Transform + metrics computation
└────┬──────────────────┘
     │ evaluation.json
┌────▼──────────────────┐
│ Step 4: Quality Gate  │  ConditionStep: Weighted F1 ≥ 0.75?
└────┬──────────────────┘
   Yes│           No→ Pipeline ends (model not promoted)
┌────▼──────────────────┐
│ Step 5: Register      │  Model Registry (Approved)
└────┬──────────────────┘
     │
┌────▼──────────────────┐
│ Step 6: Deploy        │  Lambda step → update endpoint
└────┬──────────────────┘
     │
┌────▼──────────────────┐
│ Step 7: Monitor       │  Verify monitoring schedules active
└───────────────────────┘
```

## 1. Setup & Imports

In [1]:
import boto3
import sagemaker
import json
import time
import warnings
warnings.filterwarnings('ignore')

from sagemaker import get_execution_role
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

# Pipeline components
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.parameters import ParameterString, ParameterFloat, ParameterInteger
from sagemaker.workflow.properties import PropertyFile
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor

region = boto3.Session().region_name
role = get_execution_role()
sagemaker_session = sagemaker.Session()
sm_client = boto3.client('sagemaker', region_name=region)

RAW_BUCKET = 'skyaware-raw-data'
PROCESSED_BUCKET = 'skyaware-processed-data'
MODEL_BUCKET = 'skyaware-model-artifacts'
MONITORING_BUCKET = 'skyaware-logs-monitoring'
PIPELINE_NAME = 'SkyAware-MLOps-Pipeline'
MODEL_GROUP_NAME = 'SkyAwareFlightDelay'
ENDPOINT_NAME = 'skyaware-delay-endpoint'

xgb_image_uri = sagemaker.image_uris.retrieve('xgboost', region, '1.7-1')

print(f'Region: {region}')
print(f'Pipeline name: {PIPELINE_NAME}')
print(f'SageMaker SDK: {sagemaker.__version__}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


Region: us-east-1
Pipeline name: SkyAware-MLOps-Pipeline
SageMaker SDK: 2.224.4


## 2. Pipeline Parameters

In [2]:
# Parameters allow changing values at execution time without editing the pipeline definition

training_instance_type = ParameterString(
    name='TrainingInstanceType',
    default_value='ml.m5.xlarge'
)

processing_instance_type = ParameterString(
    name='ProcessingInstanceType',
    default_value='ml.m5.large'
)

min_f1_threshold = ParameterFloat(
    name='MinF1Threshold',
    default_value=0.75
)

num_training_rounds = ParameterInteger(
    name='NumTrainingRounds',
    default_value=200
)

print('Pipeline parameters defined:')
print(f'  TrainingInstanceType: ml.m5.xlarge (default)')
print(f'  ProcessingInstanceType: ml.m5.large (default)')
print(f'  MinF1Threshold: 0.75 (model approval threshold)')
print(f'  NumTrainingRounds: 200 (XGBoost num_round)')

Pipeline parameters defined:
  TrainingInstanceType: ml.m5.xlarge (default)
  ProcessingInstanceType: ml.m5.large (default)
  MinF1Threshold: 0.75 (model approval threshold)
  NumTrainingRounds: 200 (XGBoost num_round)


## 3. Step 1: Data Processing

In [3]:
# SKLearnProcessor for feature engineering and temporal split
sklearn_processor = SKLearnProcessor(
    framework_version='1.0-1',
    instance_type=processing_instance_type,
    instance_count=1,
    base_job_name='skyaware-preprocessing',
    role=role,
    sagemaker_session=sagemaker_session
)

# Processing step: reads raw CSV, outputs train/val/test splits
# preprocess.py encodes carriers/airports, engineers features, writes XGB-format CSVs
step_process = ProcessingStep(
    name='SkyAwareDataProcessing',
    processor=sklearn_processor,
    inputs=[
        ProcessingInput(
            source=f's3://{RAW_BUCKET}/raw/Airline_Delay_Cause.csv',
            destination='/opt/ml/processing/input'
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name='train',
            source='/opt/ml/processing/train',
            destination=f's3://{PROCESSED_BUCKET}/pipeline/train/'
        ),
        ProcessingOutput(
            output_name='validation',
            source='/opt/ml/processing/val',
            destination=f's3://{PROCESSED_BUCKET}/pipeline/val/'
        ),
        ProcessingOutput(
            output_name='test',
            source='/opt/ml/processing/test',
            destination=f's3://{PROCESSED_BUCKET}/pipeline/test/'
        )
    ],
    code=f's3://{PROCESSED_BUCKET}/scripts/preprocess.py'
    # Note: upload preprocess.py from NB03 logic to S3 before running pipeline
)

print('Step 1 (Data Processing) defined.')
print('  Input:  s3://skyaware-raw-data/raw/Airline_Delay_Cause.csv')
print('  Output: s3://skyaware-processed-data/pipeline/{train,val,test}/')

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Step 1 (Data Processing) defined.
  Input:  s3://skyaware-raw-data/raw/Airline_Delay_Cause.csv
  Output: s3://skyaware-processed-data/pipeline/{train,val,test}/


## 4. Step 2: XGBoost Training

In [4]:
xgb_estimator = Estimator(
    image_uri=xgb_image_uri,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=f's3://{MODEL_BUCKET}/pipeline/',
    base_job_name='skyaware-xgboost-pipeline',
    sagemaker_session=sagemaker_session
)

xgb_estimator.set_hyperparameters(
    objective='multi:softmax',
    num_class=4,
    max_depth=6,
    eta=0.1,
    num_round=num_training_rounds,  # pipeline parameter
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    eval_metric='merror',
    early_stopping_rounds=20,
    seed=42
)

# TrainingStep uses ProcessingStep output property references — no hard-coded S3 paths
step_train = TrainingStep(
    name='SkyAwareXGBoostTraining',
    estimator=xgb_estimator,
    inputs={
        'train': TrainingInput(
            step_process.properties.ProcessingOutputConfig.Outputs['train'].S3Output.S3Uri,
            content_type='text/csv'
        ),
        'validation': TrainingInput(
            step_process.properties.ProcessingOutputConfig.Outputs['validation'].S3Output.S3Uri,
            content_type='text/csv'
        )
    },
    depends_on=[step_process]
)

print('Step 2 (XGBoost Training) defined.')
print('  Inputs: outputs of Step 1 (property references — resolved at runtime)')
print('  Output: model artifact S3 path')

Step 2 (XGBoost Training) defined.
  Inputs: outputs of Step 1 (property references — resolved at runtime)
  Output: model artifact S3 path


## 5. Step 3: Model Evaluation

In [5]:
from sagemaker.processing import ScriptProcessor

# Use the XGBoost container (1.7-1) for evaluation — same image as training,
# so the model loads without version mismatch issues.
eval_processor = ScriptProcessor(
    image_uri=xgb_image_uri,
    command=['python3'],
    instance_type=processing_instance_type,
    instance_count=1,
    role=role,
    sagemaker_session=sagemaker_session,
    base_job_name='skyaware-evaluation'
)

# evaluation.json is written by evaluate.py — contains weighted_f1 score
evaluation_report = PropertyFile(
    name='EvaluationReport',
    output_name='evaluation',
    path='evaluation.json'
)

step_eval = ProcessingStep(
    name='SkyAwareModelEvaluation',
    processor=eval_processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination='/opt/ml/processing/model'
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs['test'].S3Output.S3Uri,
            destination='/opt/ml/processing/test'
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name='evaluation',
            source='/opt/ml/processing/evaluation',
            destination=f's3://{MODEL_BUCKET}/pipeline/evaluation/'
        )
    ],
    code=f's3://{PROCESSED_BUCKET}/scripts/evaluate.py',
    property_files=[evaluation_report],
    depends_on=[step_train]
)

print('Step 3 (Model Evaluation) defined.')
print('  Processor: XGBoost 1.7-1 container (matches training container)')
print('  Inputs: model artifact + test split')
print('  Outputs: evaluation.json (weighted F1 score)')
print('  PropertyFile: EvaluationReport — parsed by ConditionStep')

Step 3 (Model Evaluation) defined.
  Processor: XGBoost 1.7-1 container (matches training container)
  Inputs: model artifact + test split
  Outputs: evaluation.json (weighted F1 score)
  PropertyFile: EvaluationReport — parsed by ConditionStep


## 6. Step 4: Quality Gate (Condition)

In [6]:
# Quality gate: only register model if weighted F1 >= threshold
cond_f1 = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path='classification_metrics.weighted_f1.value'
    ),
    right=min_f1_threshold  # pipeline parameter: default 0.75
)

print('Step 4 (Quality Gate) defined.')
print(f'  Condition: weighted_f1 >= MinF1Threshold (default: 0.75)')
print('  if_steps: [RegisterModel]')
print('  else_steps: [] (pipeline ends — model not promoted)')

Step 4 (Quality Gate) defined.
  Condition: weighted_f1 >= MinF1Threshold (default: 0.75)
  if_steps: [RegisterModel]
  else_steps: [] (pipeline ends — model not promoted)


## 7. Step 5: Register Model

In [7]:
step_register = RegisterModel(
    name='SkyAwareRegisterModel',
    estimator=xgb_estimator,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.m5.large', 'ml.m5.xlarge'],
    transform_instances=['ml.m5.xlarge', 'ml.m5.2xlarge'],
    model_package_group_name=MODEL_GROUP_NAME,
    approval_status='Approved',
    model_metrics=sagemaker.model_metrics.ModelMetrics(
        model_statistics=sagemaker.model_metrics.MetricsSource(
            s3_uri=Join(
                on='/',
                values=[
                    step_eval.properties.ProcessingOutputConfig.Outputs['evaluation'].S3Output.S3Uri,
                    'evaluation.json'
                ]
            ),
            content_type='application/json'
        )
    )
)

print('Step 5 (Register Model) defined.')
print(f'  Model group: {MODEL_GROUP_NAME}')
print('  Approval: Approved (auto-approved if F1 gate passes)')

Step 5 (Register Model) defined.
  Model group: SkyAwareFlightDelay
  Approval: Approved (auto-approved if F1 gate passes)


## 8. Step 6: Deploy Endpoint (Lambda Step)

In [8]:
# Step 6: Deploy/update the SageMaker endpoint via a Processing script
# Using ProcessingStep instead of LambdaStep — deploy_endpoint.py calls SageMaker APIs
# internally via boto3, so no separate Lambda function deployment is required.
step_deploy = ProcessingStep(
    name='SkyAwareDeployEndpoint',
    processor=sklearn_processor,
    inputs=[],
    outputs=[],
    code=f's3://{PROCESSED_BUCKET}/scripts/deploy_endpoint.py',
    job_arguments=[
        '--model-package-group', MODEL_GROUP_NAME,
        '--endpoint-name', ENDPOINT_NAME,
        '--instance-type', 'ml.m5.large',
        '--data-capture-bucket', MONITORING_BUCKET,
        '--role-arn', role,
        '--region', region
    ]
    # depends_on omitted: deploy_endpoint.py polls for the newly registered model package
    # at runtime, so it self-synchronises without requiring a DAG dependency on RegisterModel
)

print('Step 6 (Deploy Endpoint) defined as ProcessingStep.')
print(f'  Script: s3://{PROCESSED_BUCKET}/scripts/deploy_endpoint.py')
print(f'  Target endpoint: {ENDPOINT_NAME}')
print('  Polls for approved model package, then creates/updates endpoint')

Step 6 (Deploy Endpoint) defined as ProcessingStep.
  Script: s3://skyaware-processed-data/scripts/deploy_endpoint.py
  Target endpoint: skyaware-delay-endpoint
  Polls for approved model package, then creates/updates endpoint


## 9. Step 7: Verify Monitoring

In [9]:
# Final ProcessingStep — runs a script that checks monitoring schedules exist
# and creates them if missing (idempotent)
step_monitor_verify = ProcessingStep(
    name='SkyAwareMonitoringSetup',
    processor=sklearn_processor,
    inputs=[],
    outputs=[],
    code=f's3://{PROCESSED_BUCKET}/scripts/setup_monitoring.py',
    job_arguments=[
        '--endpoint-name', ENDPOINT_NAME,
        '--monitoring-bucket', MONITORING_BUCKET,
        '--region', region
    ],
    depends_on=[step_deploy]
)

print('Step 7 (Monitoring Verification) defined.')
print(f'  Verifies monitoring schedules for: {ENDPOINT_NAME}')
print('  Idempotent: safe to run multiple times')

Step 7 (Monitoring Verification) defined.
  Verifies monitoring schedules for: skyaware-delay-endpoint
  Idempotent: safe to run multiple times


## 9.5 Prerequisites: Upload Processing Scripts to S3

The pipeline references four Python scripts that must exist in S3 before execution:

| Script | S3 Key | Used By |
|---|---|---|
| `preprocess.py` | `scripts/preprocess.py` | Step 1 — feature engineering + temporal split |
| `evaluate.py` | `scripts/evaluate.py` | Step 3 — weighted F1 on test set |
| `deploy_endpoint.py` | `scripts/deploy_endpoint.py` | Step 6 — create/update SageMaker endpoint |
| `setup_monitoring.py` | `scripts/setup_monitoring.py` | Step 7 — verify monitoring schedules |

Run the cell below **once** to write and upload all four scripts. The cell is idempotent — safe to re-run.

In [10]:
PREPROCESS_PY = '''
import os, io, pickle, boto3
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def main():
    input_dir = "/opt/ml/processing/input"
    train_dir = "/opt/ml/processing/train"
    val_dir   = "/opt/ml/processing/val"
    test_dir  = "/opt/ml/processing/test"
    for d in [train_dir, val_dir, test_dir]:
        os.makedirs(d, exist_ok=True)

    csv_files = [f for f in os.listdir(input_dir) if f.endswith(".csv")]
    df = pd.read_csv(os.path.join(input_dir, csv_files[0]))
    df.columns = df.columns.str.strip()
    df = df[df["year"] >= 2010].copy()
    df = df[df["arr_flights"] > 0].copy()

    delay_cols = ["arr_del15","carrier_ct","weather_ct","nas_ct","security_ct",
                  "late_aircraft_ct","arr_cancelled","arr_diverted","arr_delay",
                  "carrier_delay","weather_delay","nas_delay","security_delay","late_aircraft_delay"]
    df[delay_cols] = df[delay_cols].fillna(0)

    df["delay_rate"]  = df["arr_del15"]     / df["arr_flights"]
    df["cancel_rate"] = df["arr_cancelled"] / df["arr_flights"]
    df["divert_rate"] = df["arr_diverted"]  / df["arr_flights"]
    df["is_covid_period"]    = df["year"].isin([2020,2021]).astype(int)
    df["late_aircraft_rate"] = df["late_aircraft_ct"] / df["arr_flights"]
    df["carrier_delay_rate"] = df["carrier_ct"]       / df["arr_flights"]
    df["weather_delay_rate"] = df["weather_ct"]        / df["arr_flights"]
    df["nas_delay_rate"]     = df["nas_ct"]            / df["arr_flights"]
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

    total_ct = df[["carrier_ct","weather_ct","nas_ct","security_ct","late_aircraft_ct"]].sum(axis=1).clip(lower=1)
    df["carrier_delay_pct"]  = df["carrier_ct"]      / total_ct
    df["weather_delay_pct"]  = df["weather_ct"]       / total_ct
    df["nas_delay_pct"]      = df["nas_ct"]           / total_ct
    df["security_delay_pct"] = df["security_ct"]      / total_ct
    df["late_aircraft_pct"]  = df["late_aircraft_ct"] / total_ct
    df["avg_delay_per_flight"] = df["arr_delay"] / df["arr_flights"].clip(lower=1)
    df["quarter"]        = ((df["month"]-1)//3+1).astype(int)
    df["is_winter"]      = df["month"].isin([12,1,2]).astype(int)
    df["is_summer"]      = df["month"].isin([6,7,8]).astype(int)
    df["is_peak_travel"] = df["month"].isin([6,7,8,11,12]).astype(int)

    df = df.sort_values(["carrier","year","month"]).reset_index(drop=True)
    df["carrier_rolling_delay_3m"]  = df.groupby("carrier")["delay_rate"].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    df["carrier_rolling_delay_12m"] = df.groupby("carrier")["delay_rate"].transform(lambda x: x.shift(1).rolling(12,min_periods=1).mean())
    df["late_aircraft_roll_3m"]     = df.groupby("carrier")["late_aircraft_rate"].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    df = df.sort_values(["airport","year","month"]).reset_index(drop=True)
    df["airport_rolling_delay_3m"]  = df.groupby("airport")["delay_rate"].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    df = df.sort_values(["carrier","year","month"]).reset_index(drop=True)
    df["carrier_yoy_delay_change"]  = df.groupby(["carrier","month"])["delay_rate"].transform(lambda x: x.diff())
    for c in ["carrier_rolling_delay_3m","carrier_rolling_delay_12m","late_aircraft_roll_3m",
              "airport_rolling_delay_3m","carrier_yoy_delay_change"]:
        df[c] = df[c].fillna(0)

    def assign_label(row):
        if row["cancel_rate"] > 0.05 or row["delay_rate"] > 0.40: return 3
        elif row["delay_rate"] > 0.25: return 2
        elif row["delay_rate"] > 0.15: return 1
        else: return 0
    df["delay_risk_label"] = df.apply(assign_label, axis=1)

    train_raw = df[df["year"] <= 2018].copy()
    val_raw   = df[(df["year"] >= 2019) & (df["year"] <= 2021)].copy()
    test_raw  = df[df["year"] >= 2022].copy()
    print(f"Train: {len(train_raw)}, Val: {len(val_raw)}, Test: {len(test_raw)}")

    carrier_enc = LabelEncoder().fit(train_raw["carrier"])
    airport_enc = LabelEncoder().fit(train_raw["airport"])

    FEATURE_COLS = [
        "year","month","month_sin","month_cos","quarter","is_winter","is_summer",
        "is_peak_travel","is_covid_period","arr_flights","delay_rate","cancel_rate",
        "divert_rate","avg_delay_per_flight","late_aircraft_rate","carrier_delay_rate",
        "weather_delay_rate","nas_delay_rate","carrier_delay_pct","weather_delay_pct",
        "nas_delay_pct","security_delay_pct","late_aircraft_pct",
        "carrier_rolling_delay_3m","carrier_rolling_delay_12m","late_aircraft_roll_3m",
        "airport_rolling_delay_3m","carrier_yoy_delay_change","carrier_enc","airport_enc",
    ]
    TARGET = "delay_risk_label"

    def encode_and_export(raw, out_path, filename):
        out = raw.copy()
        out.loc[~out["carrier"].isin(carrier_enc.classes_), "carrier"] = carrier_enc.classes_[0]
        out.loc[~out["airport"].isin(airport_enc.classes_), "airport"] = airport_enc.classes_[0]
        out["carrier_enc"] = carrier_enc.transform(out["carrier"])
        out["airport_enc"] = airport_enc.transform(out["airport"])
        avail = [c for c in FEATURE_COLS if c in out.columns]
        result = out[avail + [TARGET]].copy()
        result.replace([np.inf,-np.inf], np.nan, inplace=True)
        result.fillna(0, inplace=True)
        xgb_df = pd.concat([result[[TARGET]], result[avail]], axis=1)
        xgb_df.to_csv(os.path.join(out_path, filename), index=False, header=False)
        print(f"Wrote {len(xgb_df)} rows -> {out_path}/{filename}")

    encode_and_export(train_raw, train_dir, "train.csv")
    encode_and_export(val_raw,   val_dir,   "val.csv")
    encode_and_export(test_raw,  test_dir,  "test.csv")

    s3 = boto3.client("s3")
    for enc, name in [(carrier_enc,"carrier_encoder"),(airport_enc,"airport_encoder")]:
        buf = io.BytesIO(); pickle.dump(enc, buf); buf.seek(0)
        s3.put_object(Bucket="skyaware-processed-data", Key=f"encoders/{name}.pkl", Body=buf.read())
        print(f"Saved encoder: s3://skyaware-processed-data/encoders/{name}.pkl")

if __name__ == "__main__":
    main()
'''

EVALUATE_PY = '''
import os, json, tarfile, pickle
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report

def find_and_load_model(extract_dir):
    all_files = []
    for root, dirs, files in os.walk(extract_dir):
        for f in files:
            all_files.append(os.path.join(root, f))
    print(f"Files extracted from model.tar.gz: {all_files}")

    candidates = ["xgboost-model", "model", "model.bin", "model.xgb", "model.pkl"]
    for candidate in candidates:
        for fpath in all_files:
            if os.path.basename(fpath) == candidate:
                try:
                    b = xgb.Booster()
                    b.load_model(fpath)
                    print(f"Loaded XGBoost binary model: {fpath}")
                    return b
                except Exception as e1:
                    print(f"  xgb.Booster.load_model failed: {e1}")
                try:
                    with open(fpath, "rb") as fh:
                        m = pickle.load(fh)
                    if hasattr(m, "predict"):
                        print(f"Loaded pickle model: {fpath}")
                        return m
                except Exception as e2:
                    print(f"  pickle.load failed: {e2}")

    for fpath in all_files:
        try:
            b = xgb.Booster()
            b.load_model(fpath)
            print(f"Loaded XGBoost model (fallback): {fpath}")
            return b
        except Exception:
            pass
        try:
            with open(fpath, "rb") as fh:
                m = pickle.load(fh)
            if hasattr(m, "predict"):
                print(f"Loaded pickle model (fallback): {fpath}")
                return m
        except Exception:
            pass

    raise RuntimeError(f"Could not load model from any file. Extracted: {all_files}")


def main():
    model_dir = "/opt/ml/processing/model"
    test_dir  = "/opt/ml/processing/test"
    eval_dir  = "/opt/ml/processing/evaluation"
    os.makedirs(eval_dir, exist_ok=True)

    print(f"XGBoost version: {xgb.__version__}")
    print(f"Model dir contents: {os.listdir(model_dir)}")

    extract_dir = "/tmp/model_extract"
    os.makedirs(extract_dir, exist_ok=True)

    tgz_path = os.path.join(model_dir, "model.tar.gz")
    with tarfile.open(tgz_path, "r:gz") as tar:
        tar.extractall(extract_dir)

    model = find_and_load_model(extract_dir)

    test_csvs = sorted([os.path.join(test_dir, f)
                        for f in os.listdir(test_dir) if f.endswith(".csv")])
    print(f"Test files: {test_csvs}")
    test_df = pd.concat([pd.read_csv(f, header=None) for f in test_csvs], ignore_index=True)
    print(f"Test shape: {test_df.shape}")

    y_true = test_df.iloc[:, 0].astype(int).values
    X      = test_df.iloc[:, 1:].values

    if isinstance(model, xgb.Booster):
        y_pred = model.predict(xgb.DMatrix(X)).astype(int)
    else:
        y_pred = model.predict(X).astype(int)

    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    print(f"Test samples: {len(y_true):,}")
    print(f"Weighted F1:  {weighted_f1:.4f}")
    print(classification_report(y_true, y_pred,
          target_names=["on-time","minor-risk","major-risk","high-risk"]))

    evaluation = {"classification_metrics": {"weighted_f1": {"value": float(weighted_f1)}}}
    out_path = os.path.join(eval_dir, "evaluation.json")
    with open(out_path, "w") as fh:
        json.dump(evaluation, fh, indent=2)
    print(f"Saved: {out_path}")
    print(json.dumps(evaluation, indent=2))

if __name__ == "__main__":
    main()
'''

# deploy_endpoint.py — SKLearnProcessor container has no sagemaker SDK and no
# AWS_DEFAULT_REGION. Role ARN and region are passed explicitly as job arguments.
DEPLOY_ENDPOINT_PY = '''
import argparse, time, boto3

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-package-group", required=True)
    parser.add_argument("--endpoint-name",        required=True)
    parser.add_argument("--instance-type",        default="ml.m5.large")
    parser.add_argument("--data-capture-bucket",  required=True)
    parser.add_argument("--role-arn",             required=True)
    parser.add_argument("--region",               required=True)
    args = parser.parse_args()

    sm   = boto3.client("sagemaker", region_name=args.region)
    role = args.role_arn
    print(f"Region: {args.region} | Endpoint: {args.endpoint_name} | Group: {args.model_package_group}")

    pkg_arn, waited = None, 0
    while waited < 300:
        pkgs = sm.list_model_packages(
            ModelPackageGroupName=args.model_package_group,
            ModelApprovalStatus="Approved",
            SortBy="CreationTime", SortOrder="Descending"
        )["ModelPackageSummaryList"]
        if pkgs:
            pkg_arn = pkgs[0]["ModelPackageArn"]
            print(f"Model package: {pkg_arn}")
            break
        print(f"Waiting for model package... ({waited}s)")
        time.sleep(30); waited += 30

    if pkg_arn is None:
        raise RuntimeError("No approved model packages found after 5 minutes")

    ts = str(int(time.time()))
    model_name  = f"skyaware-model-{ts}"
    config_name = f"skyaware-ep-config-{ts}"

    sm.create_model(
        ModelName=model_name,
        Containers=[{"ModelPackageName": pkg_arn}],
        ExecutionRoleArn=role
    )
    print(f"Created model: {model_name}")

    sm.create_endpoint_config(
        EndpointConfigName=config_name,
        ProductionVariants=[{
            "VariantName": "primary",
            "ModelName": model_name,
            "InstanceType": args.instance_type,
            "InitialInstanceCount": 1,
            "InitialVariantWeight": 1.0
        }],
        DataCaptureConfig={
            "EnableCapture": True,
            "InitialSamplingPercentage": 20,
            "DestinationS3Uri": f"s3://{args.data_capture_bucket}/data-capture/",
            "CaptureOptions": [{"CaptureMode":"Input"},{"CaptureMode":"Output"}]
        }
    )
    print(f"Created endpoint config: {config_name}")

    try:
        sm.describe_endpoint(EndpointName=args.endpoint_name)
        sm.update_endpoint(EndpointName=args.endpoint_name, EndpointConfigName=config_name)
        print("Updating existing endpoint...")
    except sm.exceptions.from_code("ValidationException"):
        sm.create_endpoint(EndpointName=args.endpoint_name, EndpointConfigName=config_name)
        print("Creating new endpoint...")

    print("Waiting for endpoint InService (up to 20 min)...")
    waiter = sm.get_waiter("endpoint_in_service")
    waiter.wait(EndpointName=args.endpoint_name,
                WaiterConfig={"Delay":30,"MaxAttempts":40})
    status = sm.describe_endpoint(EndpointName=args.endpoint_name)["EndpointStatus"]
    print(f"Endpoint status: {status}")

if __name__ == "__main__":
    main()
'''

# setup_monitoring.py — same fix: region passed as argument, not from environment
SETUP_MONITORING_PY = '''
import argparse, time, boto3

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--endpoint-name",     required=True)
    parser.add_argument("--monitoring-bucket", required=True)
    parser.add_argument("--region",            required=True)
    args = parser.parse_args()

    sm = boto3.client("sagemaker", region_name=args.region)
    print(f"Checking monitoring for: {args.endpoint_name} in {args.region}")

    waited = 0
    while waited < 600:
        try:
            status = sm.describe_endpoint(EndpointName=args.endpoint_name)["EndpointStatus"]
            if status == "InService":
                print(f"Endpoint InService")
                break
            elif status in ("Failed","OutOfService"):
                print(f"Endpoint status: {status} — monitoring check skipped")
                return
            print(f"Endpoint status: {status}, waiting...")
        except Exception as e:
            print(f"Endpoint not found: {e}")
            return
        time.sleep(30); waited += 30

    try:
        schedules = sm.list_monitoring_schedules(
            EndpointName=args.endpoint_name
        )["MonitoringScheduleSummaries"]
        if schedules:
            print(f"Active monitoring schedules ({len(schedules)}):")
            for s in schedules:
                print(f"  {s['MonitoringScheduleName']}: {s['MonitoringScheduleStatus']}")
        else:
            print("No monitoring schedules found — set them up via SageMaker Studio or NB07.")
    except Exception as e:
        print(f"Monitoring schedule check: {e}")

    print("Monitoring verification complete.")

if __name__ == "__main__":
    main()
'''

# Upload all four scripts to S3
s3 = boto3.client('s3', region_name=region)

scripts = {
    'preprocess.py':       PREPROCESS_PY,
    'evaluate.py':         EVALUATE_PY,
    'deploy_endpoint.py':  DEPLOY_ENDPOINT_PY,
    'setup_monitoring.py': SETUP_MONITORING_PY,
}

for name, content in scripts.items():
    s3.put_object(
        Bucket=PROCESSED_BUCKET,
        Key=f'scripts/{name}',
        Body=content.strip().encode('utf-8')
    )
    print(f'Uploaded s3://{PROCESSED_BUCKET}/scripts/{name}')

print('\nAll scripts uploaded. Pipeline is ready to execute.')

Uploaded s3://skyaware-processed-data/scripts/preprocess.py
Uploaded s3://skyaware-processed-data/scripts/evaluate.py
Uploaded s3://skyaware-processed-data/scripts/deploy_endpoint.py
Uploaded s3://skyaware-processed-data/scripts/setup_monitoring.py

All scripts uploaded. Pipeline is ready to execute.


## 10. Assemble & Submit Pipeline

In [11]:
# Build the ConditionStep with both branches defined
step_cond = ConditionStep(
    name='SkyAwareF1QualityGate',
    conditions=[cond_f1],
    if_steps=[step_register, step_deploy, step_monitor_verify],
    else_steps=[]
)

# Full pipeline definition
pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=[
        training_instance_type,
        processing_instance_type,
        min_f1_threshold,
        num_training_rounds
    ],
    steps=[step_process, step_train, step_eval, step_cond],
    sagemaker_session=sagemaker_session
)

print('Pipeline assembled.')
print(f'Name: {PIPELINE_NAME}')
print(f'Steps: {[s.name for s in [step_process, step_train, step_eval, step_cond]]}')

Pipeline assembled.
Name: SkyAware-MLOps-Pipeline
Steps: ['SkyAwareDataProcessing', 'SkyAwareXGBoostTraining', 'SkyAwareModelEvaluation', 'SkyAwareF1QualityGate']


In [12]:
# Upsert (create or update) the pipeline definition
try:
    pipeline_arn = pipeline.upsert(role_arn=role)
    print(f'Pipeline upserted successfully!')
    print(f'ARN: {pipeline_arn["PipelineArn"]}')
except Exception as e:
    print(f'Pipeline upsert note: {e}')

# Start a pipeline execution
try:
    execution = pipeline.start(
        parameters={
            'MinF1Threshold': 0.75,
            'NumTrainingRounds': 200
        }
    )
    print(f'\nPipeline execution started!')
    print(f'Execution ARN: {execution.arn}')
except Exception as e:
    print(f'Execution start note: {e}')

Pipeline upserted successfully!
ARN: arn:aws:sagemaker:us-east-1:706029888011:pipeline/SkyAware-MLOps-Pipeline

Pipeline execution started!
Execution ARN: arn:aws:sagemaker:us-east-1:706029888011:pipeline/SkyAware-MLOps-Pipeline/execution/7h86wopfjoo6


## 11. Monitor Pipeline Execution

In [14]:
# Poll execution status — list_steps() returns a plain list in SDK 2.x
try:
    print(f'Pipeline execution: {execution.arn}')
    print('\nStep statuses:')
    steps = execution.list_steps()
    step_list = steps if isinstance(steps, list) else steps.get('PipelineExecutionSteps', steps)
    for step in step_list:
        name   = step.get('StepName', step.get('Name', '?'))
        status = step.get('StepStatus', step.get('Status', '?'))
        print(f'  {name:<40} {status}')
    exec_status = execution.describe()['PipelineExecutionStatus']
    print(f'\nOverall status: {exec_status}')
except Exception as e:
    print(f'Status check note: {e}')

Pipeline execution: arn:aws:sagemaker:us-east-1:706029888011:pipeline/SkyAware-MLOps-Pipeline/execution/7h86wopfjoo6

Step statuses:
  SkyAwareMonitoringSetup                  Succeeded
  SkyAwareDeployEndpoint                   Succeeded
  SkyAwareRegisterModel-RegisterModel      Succeeded
  SkyAwareF1QualityGate                    Succeeded
  SkyAwareModelEvaluation                  Succeeded
  SkyAwareXGBoostTraining                  Succeeded
  SkyAwareDataProcessing                   Succeeded



Overall status: Succeeded


In [15]:
# Optional: wait for pipeline to complete (blocks kernel)
# execution.wait()
# print('Pipeline complete!')

# List all pipeline executions
try:
    executions = sm_client.list_pipeline_executions(PipelineName=PIPELINE_NAME)
    print(f'Pipeline executions for {PIPELINE_NAME}:')
    for ex in executions['PipelineExecutionSummaries']:
        print(f'  {ex["PipelineExecutionArn"].split("/")[-1][:20]:<25} {ex["PipelineExecutionStatus"]}')
except Exception as e:
    print(f'Execution list note: {e}')

Pipeline executions for SkyAware-MLOps-Pipeline:
  7h86wopfjoo6              Succeeded
  ejuxyg547w7t              Stopped
  pts27ssl8afp              Failed
  2g5f6smz0gg5              Failed
  dexdm6hf6x3g              Failed
  jsv0b2hz2raa              Failed
  gz2zj6iy0w45              Failed
  fm8fgbzetbpa              Failed


## 12. Pipeline Trigger Options

In [16]:
# === EventBridge Rule: Monthly retraining trigger ===
events_client = boto3.client('events', region_name=region)

eb_rule_payload = {
    'RuleName': 'SkyAware-Monthly-Retraining',
    'Description': 'Trigger SkyAware pipeline on the 1st of every month at 2am UTC',
    'ScheduleExpression': 'cron(0 2 1 * ? *)',
    'State': 'ENABLED',
    # Target would be an EventBridge-to-Lambda function that calls pipeline.start()
}

print('EventBridge trigger (monthly retraining):')
print(json.dumps(eb_rule_payload, indent=2))

print('\nOther trigger options:')
print('  1. S3 Event: new data uploaded to s3://skyaware-raw-data/raw/ → Lambda → pipeline.start()')
print('  2. Manual: pipeline.start(parameters={...}) from this notebook')
print('  3. PSI alert: CloudWatch Alarm → SNS → Lambda → pipeline.start()')

EventBridge trigger (monthly retraining):
{
  "RuleName": "SkyAware-Monthly-Retraining",
  "Description": "Trigger SkyAware pipeline on the 1st of every month at 2am UTC",
  "ScheduleExpression": "cron(0 2 1 * ? *)",
  "State": "ENABLED"
}

Other trigger options:
  1. S3 Event: new data uploaded to s3://skyaware-raw-data/raw/ → Lambda → pipeline.start()
  2. Manual: pipeline.start(parameters={...}) from this notebook
  3. PSI alert: CloudWatch Alarm → SNS → Lambda → pipeline.start()


In [17]:
print('=== PIPELINE CONFIGURATION SUMMARY ===')
print(f'  Pipeline name:     {PIPELINE_NAME}')
print(f'  Model group:       {MODEL_GROUP_NAME}')
print(f'  Endpoint name:     {ENDPOINT_NAME}')
print(f'  Raw data:          s3://{RAW_BUCKET}/raw/')
print(f'  Processed data:    s3://{PROCESSED_BUCKET}/pipeline/')
print(f'  Model artifacts:   s3://{MODEL_BUCKET}/pipeline/')
print(f'  Monitoring:        s3://{MONITORING_BUCKET}/')
print(f'  F1 threshold:      0.75 (configurable via MinF1Threshold parameter)')
print(f'  Trigger:           Monthly (EventBridge) + Manual + S3 event')
print('\nPipeline ready for execution!')

=== PIPELINE CONFIGURATION SUMMARY ===
  Pipeline name:     SkyAware-MLOps-Pipeline
  Model group:       SkyAwareFlightDelay
  Endpoint name:     skyaware-delay-endpoint
  Raw data:          s3://skyaware-raw-data/raw/
  Processed data:    s3://skyaware-processed-data/pipeline/
  Model artifacts:   s3://skyaware-model-artifacts/pipeline/
  Monitoring:        s3://skyaware-logs-monitoring/
  F1 threshold:      0.75 (configurable via MinF1Threshold parameter)
  Trigger:           Monthly (EventBridge) + Manual + S3 event

Pipeline ready for execution!
